In [ ]:
# ── PACKAGES ──────────────────────────────────────
required <- c("ggplot2", "dplyr", "tidyr", "patchwork", "scales")
invisible(lapply(required, function(p) {
  if (!requireNamespace(p, quietly = TRUE))
    install.packages(p, repos = "https://cloud.r-project.org")
  library(p, character.only = TRUE)
}))

# ── CONFIG ────────────────────────────────────────────────────────────────────
ANN_CSV <- "Pathway_Annotation_50Runs.csv"
VAL_CSV <- "Pathway_Validation_50Runs.csv"
OUTDIR  <- "."
GENESET <- "Gene Set 73"

# ── PALETTE ───────────────────────────────────────────────────────────────────
col_enrich   <- "#a50026"   
col_noenrich <- "#0571b0"   
col_final    <- "#4dac26"   
col_other    <- "grey85"    

biolord <- c(
  "With Enrichment"                 = 0.9774,
  "Without Enrichment"              = 0.9215,
  "Final Process (Post-Validation)" = 0.9215
)

# ── THEME ─────────────────────────────────────────────────────────────
theme_nature <- function(base_size = 11) {
  theme_classic(base_size = base_size, base_family = "Helvetica") +
    theme(
      panel.border       = element_rect(colour = "black", fill = NA, linewidth = 0.6),
      panel.grid.major.x = element_line(colour = "grey92", linewidth = 0.35),
      panel.grid.minor   = element_blank(),
      axis.line          = element_blank(),
      axis.ticks         = element_line(colour = "black", linewidth = 0.45),
      axis.ticks.length  = unit(3, "pt"),
      axis.title         = element_text(face = "bold", size = base_size),
      axis.text          = element_text(colour = "black", size = base_size - 1),
      legend.title       = element_text(face = "bold", size = base_size - 1),
      legend.text        = element_text(size = base_size - 2),
      legend.key         = element_blank(),
      legend.background  = element_blank(),
      plot.title         = element_text(face = "bold", size = base_size + 1, hjust = 0),
      plot.subtitle      = element_text(size = base_size - 2, colour = "grey45", hjust = 0),
      strip.background   = element_rect(fill = "grey96", colour = "black", linewidth = 0.5),
      strip.text         = element_text(face = "bold", size = base_size),
      plot.margin        = margin(8, 16, 8, 8)
    )
}

theme_void_donut <- function(base_size = 9) {
  theme_void(base_size = base_size) +
    theme(legend.position = "none", plot.margin = margin(2, 2, 2, 2))
}

ann <- read.csv(ANN_CSV, stringsAsFactors = FALSE)
val <- read.csv(VAL_CSV, stringsAsFactors = FALSE)
cat(sprintf("Annotation rows: %d  |  Validation rows: %d\n", nrow(ann), nrow(val)))

cat("\n=== ANNOTATION LABEL STABILITY ===\n")
for (info in list(
  list(name = "With Enrichment (Ann)",    col = ann$Process_With_Enrichment),
  list(name = "Without Enrichment (Ann)", col = ann$Process_Without_Enrichment),
  list(name = "Final Process (Post-Val)", col = val$Final_Process)
)) {
  tbl <- sort(table(info$col), decreasing = TRUE)
  cat(sprintf("\n  %s\n    Unique labels: %d\n    Top label: '%s' x %d (%.1f%%)\n",
              info$name, length(tbl),
              names(tbl)[1], tbl[[1]], 100 * tbl[[1]] / sum(tbl)))
}

cat("\n=== CONFIDENCE STABILITY ===\n")
cat(sprintf("  %-30s  %6s  %6s  %5s  %5s  %5s\n",
            "Track", "Mean", "SD", "CV%", "Min", "Max"))
cat("  ", strrep("-", 60), "\n", sep = "")
for (info in list(
  list(name = "With Enrichment (Ann)",    v = ann$Confidence_With_Enrichment),
  list(name = "Without Enrichment (Ann)", v = ann$Confidence_Without_Enrichment),
  list(name = "Final Confidence (Val)",   v = val$Final_Confidence)
)) {
  v <- as.numeric(na.omit(info$v))
  cat(sprintf("  %-30s  %6.3f  %6.3f  %4.1f%%  %5.2f  %5.2f\n",
              info$name, mean(v), sd(v), sd(v)/mean(v)*100, min(v), max(v)))
}
cat("\n")

panel_specs <- list(
  list(texts = ann$Process_With_Enrichment,    track = "With Enrichment",
       colour = col_enrich),
  list(texts = ann$Process_Without_Enrichment, track = "Without Enrichment",
       colour = col_noenrich),
  list(texts = val$Final_Process,              track = "Final Process (Post-Validation)",
       colour = col_final)
)

make_lollipop <- function(spec) {
  texts  <- spec$texts
  clr    <- spec$colour
  track  <- spec$track
  bl_sim <- biolord[track]

  tbl <- as.data.frame(table(Label = texts), stringsAsFactors = FALSE)
  tbl <- tbl[order(tbl$Freq, decreasing = FALSE), ]
  tbl$Label  <- factor(tbl$Label, levels = tbl$Label)
  tbl$IsMode <- tbl$Freq == max(tbl$Freq)
  n_unique   <- nrow(tbl)

  inset_txt <- sprintf("Unique labels: %d\nBioLORD sim: %.1f%%", n_unique, 100 * bl_sim)

  ggplot(tbl, aes(x = Freq, y = Label)) +
    geom_segment(aes(x = 0, xend = Freq, y = Label, yend = Label,
                     colour = IsMode, linewidth = IsMode)) +
    geom_point(aes(colour = IsMode, size = IsMode)) +
    geom_text(aes(label = Freq), hjust = -0.6, size = 3.0, colour = "grey30") +
    scale_colour_manual(values = c(`TRUE` = clr, `FALSE` = alpha(clr, 0.45)), guide = "none") +
    scale_linewidth_manual(values = c(`TRUE` = 1.3, `FALSE` = 0.7), guide = "none") +
    scale_size_manual(values = c(`TRUE` = 3.6, `FALSE` = 2.0), guide = "none") +
    annotate("label",
             x = 44, y = 1.4,
             label = inset_txt,
             hjust = 0, vjust = 0,
             size = 2.9, label.size = 0.4,
             fill = "grey98", colour = clr,
             label.padding = unit(0.35, "lines")) +
    scale_x_continuous(expand = expansion(mult = c(0, 0.28)),
                       breaks = c(0, 10, 20, 30, 40, 50), limits = c(0, 55)) +
    labs(title = track, x = "Frequency (n runs)", y = NULL) +
    theme_nature(base_size = 10) +
    theme(
      panel.grid.major.y = element_blank(),
      plot.title = element_text(colour = clr, face = "bold", size = 11, hjust = 0),
      axis.text.y = element_text(size = 9, hjust = 1, margin = margin(r = 4))
    )
}

make_donut <- function(spec) {
  texts  <- spec$texts
  clr    <- spec$colour
  track  <- spec$track
  bl_sim <- biolord[track]

  tbl     <- sort(table(texts), decreasing = TRUE)
  n_uniq  <- length(tbl)
  maj_pct <- 100 * tbl[[1]] / sum(tbl)

  d <- data.frame(cat = factor(c("Majority", "Other"), levels = c("Majority", "Other")),
                  val = c(maj_pct, 100 - maj_pct))

  centre_txt <- sprintf("%d labels\n%.1f%% sim", n_uniq, 100 * bl_sim)

  ggplot(d, aes(x = 2, y = val, fill = cat)) +
    geom_col(width = 1, colour = "white", linewidth = 0.6) +
    coord_polar(theta = "y") +
    xlim(0.4, 2.5) +
    scale_fill_manual(values = c(Majority = clr, Other = col_other)) +
    annotate("text", x = 0.4, y = 0, label = centre_txt,
             size = 3.6, fontface = "bold", colour = clr, lineheight = 0.95) +
    theme_void_donut()
}

build_track_row <- function(spec) {
  lp <- make_lollipop(spec)
  dn <- make_donut(spec)
  lp + dn + plot_layout(widths = c(3.3, 1))
}

row1 <- build_track_row(panel_specs[[1]])
row2 <- build_track_row(panel_specs[[2]])
row3 <- build_track_row(panel_specs[[3]])

figA <- (row1 / row2 / row3) +
  plot_annotation(
    title    = paste("Annotation Stability Across 50 Runs —", GENESET),
    subtitle = paste(
      "Lollipop = label frequency (solid, larger dot = plurality label; faded = minority variants).",
      "Donut wedge = majority-label share of all 50 runs; centre = unique-label count & BioLORD similarity."
    ),
    theme = theme(
      plot.title    = element_text(face = "bold", size = 13, hjust = 0, family = "Helvetica"),
      plot.subtitle = element_text(size = 9, colour = "grey45", hjust = 0, family = "Helvetica"),
      plot.margin   = margin(10, 12, 6, 6)
    )
  )


conf_long <- bind_rows(
  data.frame(Track      = "With Enrichment\n(Annotation)",
             Confidence = as.numeric(ann$Confidence_With_Enrichment)),
  data.frame(Track      = "Without Enrichment\n(Annotation)",
             Confidence = as.numeric(ann$Confidence_Without_Enrichment)),
  data.frame(Track      = "Final Confidence\n(Post-Validation)",
             Confidence = as.numeric(val$Final_Confidence))
) %>%
  mutate(Track = factor(Track, levels = c(
    "With Enrichment\n(Annotation)",
    "Without Enrichment\n(Annotation)",
    "Final Confidence\n(Post-Validation)"
  )))

track_fills <- c(
  "With Enrichment\n(Annotation)"       = col_enrich,
  "Without Enrichment\n(Annotation)"    = col_noenrich,
  "Final Confidence\n(Post-Validation)" = col_final
)

conf_stats <- conf_long %>%
  group_by(Track) %>%
  summarise(
    Mean    = mean(Confidence, na.rm = TRUE),
    SD      = sd(Confidence,   na.rm = TRUE),
    CV      = sd(Confidence,   na.rm = TRUE) / mean(Confidence, na.rm = TRUE) * 100,
    Min     = min(Confidence,  na.rm = TRUE),
    Max     = max(Confidence,  na.rm = TRUE),
    .groups = "drop"
  ) %>%
  mutate(
    inset   = sprintf(
      "Mean = %.3f\nSD    = %.3f\nCV    = %.1f%%\nRange [%.2f, %.2f]",
      Mean, SD, CV, Min, Max
    ),
    y_inset = Max + 0.013
  )

y_lo <- floor(min(conf_long$Confidence, na.rm = TRUE) * 20) / 20 - 0.02
y_hi <- max(conf_stats$y_inset) + 0.075

set.seed(42)

figB <- ggplot(conf_long,
               aes(x = Track, y = Confidence,
                   fill = Track, colour = Track)) +

  geom_hline(yintercept = seq(0.70, 1.00, 0.05),
             colour = "grey90", linewidth = 0.4, linetype = "dotted") +

  # Violin — narrower width (fits tighter panel), smoother bandwidth
  geom_violin(trim = TRUE, scale = "width", width = 0.58,
              alpha = 0.32, linewidth = 0.22, colour = NA,
              adjust = 1.3) +

  # IQR box — thin, no outlier dots
  geom_boxplot(outlier.shape = NA, width = 0.10, linewidth = 0.55,
               colour = "grey25", alpha = 0.30) +

  # Jittered points — tighter horizontal spread for the narrower panel
  geom_jitter(width = 0.065, alpha = 0.85, size = 2.2,
              shape = 21, colour = "white", stroke = 0.45) +

  # Mean crossbar
  stat_summary(fun = mean, geom = "crossbar",
               width = 0.34, linewidth = 0.9,
               colour = "black", fatten = 0) +

  # ±1 SD dashed whiskers
  stat_summary(
    fun.data = function(x)
      data.frame(y    = mean(x),
                 ymin = mean(x) - sd(x),
                 ymax = mean(x) + sd(x)),
    geom      = "errorbar",
    width     = 0.20,
    linewidth = 0.75,
    linetype  = "dashed",
    colour    = "grey30"
  ) +

  # Stats inset above the cloud
  geom_label(
    data  = conf_stats,
    aes(x = Track, y = y_inset, label = inset),
    vjust = 0, hjust = 0.5,
    size  = 2.9, label.size = 0.35,
    fill  = "grey98", colour = "grey15",
    fontface = "plain",
    label.padding = unit(0.38, "lines"),
    inherit.aes = FALSE
  ) +

  scale_fill_manual(values   = track_fills, guide = "none") +
  scale_colour_manual(values = track_fills, guide = "none") +
  scale_y_continuous(
    breaks = seq(0.70, 1.00, 0.05),
    limits = c(y_lo, y_hi),
    expand = expansion(mult = c(0.02, 0.01))
  ) +
  labs(
    title    = paste("Confidence Score Stability Across 50 Runs —", GENESET),
    subtitle = "Each point = one run  |  bar = mean  |  dashed whiskers = \u00b11 SD  |  box = IQR",
    x        = NULL,
    y        = "Confidence Score"
  ) +
  theme_nature(base_size = 11) +
  theme(
    panel.grid.major.y = element_line(colour = "grey92", linewidth = 0.35),
    panel.grid.major.x = element_blank(),
    axis.text.x = element_text(
      size   = 10,
      face   = "bold",
      colour = unname(track_fills[levels(conf_long$Track)])
    )
  )

# ── SAVE ──────────────────────────────────────────────────────────────────────
dir.create(OUTDIR, showWarnings = FALSE, recursive = TRUE)

ggsave(file.path(OUTDIR, "Fig_Annotation_Stability_v4.pdf"),
       figA, width = 15, height = 13.5, dpi = 300)
ggsave(file.path(OUTDIR, "Fig_Annotation_Stability_v4.png"),
       figA, width = 15, height = 13.5, dpi = 300)

ggsave(file.path(OUTDIR, "Fig_Confidence_Stability_v4.pdf"),
       figB, width = 8.5, height = 6.8, dpi = 300)
ggsave(file.path(OUTDIR, "Fig_Confidence_Stability_v4.png"),
       figB, width = 8.5, height = 6.8, dpi = 300)

cat("Saved:\n")
cat("  Fig_Annotation_Stability_v4.pdf/.png\n")
cat("  Fig_Confidence_Stability_v4.pdf/.png\n")